# Vimeo folder to Course ouline

## Get folder items

Sử dụng Vimeo API để lấy danh sách video ở một folder trên Vimeo

In [ ]:
const options = {
    method: "GET",
    redirect: "follow",
    headers: {
        "Authorization": `Bearer ${accessToken}`,
        "Content-Type": "application/json"
    }
};
// Make a GET request to the Vimeo API
apiUrl = `https://api.vimeo.com/users/${myID}/projects/${projectID}/items`
const response = UrlFetchApp.fetch(apiUrl, options)
data = JSON.parse(response.getContentText())

## Get videos with infomation

Từ list item lấy danh sách video

In [ ]:
const videos = [];
for (const item of data.data) {
    const name = parse_name(item.video.name)
    const pk = name[0]
    const title = name[1]
    const video = {
        pk,
        url: item.video.link,
        title,
        description: item.video.description,
        duration: get_duration(item.video.duration)
    }
    videos.push(video)
}

Với tiêu đề video lấy số thự tự của video

In [ ]:
function parse_name(str) {
    const regex = /((Bài|Buổi)\s)?(\d+)(( - )|(: ))(.*)/;
    const match = str.match(regex);
    if (match) {
        const extractedNumber = parseInt(match[3], 10);
        const extractedString = match[6];
        return [extractedNumber, extractedString]
    } else return ["", str];
}

Từ output sau khi GET API sẽ phân ra từng section dựa trên Description của Video

In [ ]:
% % script node

data = [
    {
        "pk": "",
        "url": "https://vimeo.com/821909358",
        "title": "Weekly LLD Meeting 2023-04-28 02:42:37",
        "description": null,
        "duration": "55:15"
    }, {
        "pk": "",
        "url": "https://vimeo.com/821846221",
        "title": "Weekly LLD Meeting 2023-04-27 23:12:02",
        "description": "A - Laladay MEET",
        "duration": "10:30"
    }, {
        "pk": "",
        "url": "https://vimeo.com/820755680",
        "title": "Weekly LLD Meeting 2023-04-25 04:01:28",
        "description": null,
        "duration": "13:02"
    }, {
        "pk": "",
        "url": "https://vimeo.com/810021916",
        "title": "Weekly LLD Meeting 2023-03-21 03:25:21",
        "description": "B",
        "duration": "30:12"
    }
]


function videos2outline(videos) {
    const groupedVideos = videos.reduce((acc, video) => {
        const description = video.description
        if (!acc[description]) {
            acc[description] = [];
        }
        acc[description].push({
            title: video.title,
            url: video.url,
            duration: video.duration,
        });
        return acc;
    }, {});

    const sections = Object.keys(groupedVideos).map((description) => ({
        heading: description.split(" - ")[0] || 'General',
        description: description.split(" - ")[1] || '', // You can customize this if needed
        lessons: groupedVideos[description],
    }));

    return JSON.stringify(sections, null, 2);
}
HTML = `document.currentScript.setAttribute("outline", JSON.stringify(` + videos2outline(data) + `))`;
console.log(HTML);

In [ ]:
%%script node

const url = 'https://api.vimeo.com'
const sregex = /(?<pk>\d+)(.? )(?<name>.*)/;
const vregex = /(((?<pre>Bài|Buổi)\s)?(?<pk>\d+)\s?(-|:|.)?)?(?<name>.*)/;
const options = {
    method: "GET",
    redirect: "follow",
    headers: {
        "Content-Type": "application/json"
    }
};
function GET(url) {
    return fetch(url, options).then(res => res.json())
}
function getSections(id) {
    let duration_time = 0
    // const data = GET(`${url}/users/${myID}/projects/${id}/items?per_page=100&sort=alphabetical`)
    const folders = []
    for (const item of data.data) {
      if (item.type == "folder") {
        const { pk, name } = item.folder.name.match(sregex).groups;
        const api = url + item.folder.uri + '/items?per_page=100&sort=alphabetical'
        const { duration, videos }= getLessons(api)
        const folder = {
          pk,
          name,
          videos
        }
        folders.push(folder);
        duration_time += duration
      }
    }
    // return [...folders].map(f => ({ ...f, videos: getVideos(f.pk, f.videos) })).filter(f => f.videos.length > 0)
    return [folders.sort((a, b) => parseFloat(a.pk) - parseFloat(b.pk)), get_duration(duration_time)];
  }
  function getLessons(api) {
    const videos = []
    // const data = GET(api)
    let duration = 0
    for (const item of data1.data) {
      if (item.type === 'video') {
        const { pre, pk, name } = item.video.name.match(vregex).groups;
        const title = (pre != null) ? `${pre} ${pk}: ${name}` : item.video.name
        const video = {
          pk,
          url: item.video.link,
          title,
          description: item.video.description,
          duration: get_duration(item.video.duration)
        }
        duration += item.video.duration
        videos.push(video)
      }
    }
    return {videos: videos.sort((a, b) => parseFloat(a.pk) - parseFloat(b.pk)), duration};
  }
  
  function get_duration(seconds) {
    return new Date(seconds * 1000).toISOString().slice(14, 19);
  }

console.log(JSON.stringify(getSections(1)))


In [14]:
%%script node
// Function to load JSON data from a file
const fs = require('fs').promises;
const path = require('path');

async function loadJSON(file) {
  try {
    const filePath = path.resolve(__dirname, file);
    const data = await fs.readFile(filePath, 'utf8');
    return JSON.parse(data);
  } catch (error) {
    console.error('Error loading JSON file:', error);
  }
}
function get_duration(time) {
    const hours = Math.floor(time / 3600);
    const minutes = Math.floor((time % 3600) / 60);
    const seconds = time % 60;
  
    return `${hours ? pad(hours) + ':' : ''}${pad(minutes)}:${pad(seconds)}`;
  }
  
  function pad(number) {
    return (number < 10 ? '0' : '') + number;
  }
  
  
const vregex = /(((?<pre>Bài|Buổi)\s+)?(?<pk>\d+)\s?(-|:|.)?\s?)?(?<name>.*)/;

loadJSON("datas\\subfolder.json")
.then(data => {
    const videos = []
    let duration = 0
  for (const item of data.data) {
    if (item.type === 'video') {
      const { pre, pk, name } = item.video.name.match(vregex).groups;
      const title = (pre != null) ? `${pre} ${pk}: ${name}` : item.video.name
      const video = {
        pk,
        url: item.video.link,
        title,
        description: item.video.description,
        duration: get_duration(item.video.duration)
      }
      duration += item.video.duration
      videos.push(video)
    }
  }
  console.log( { lessons: videos.sort((a, b) => parseFloat(a.pk) - parseFloat(b.pk)), duration })
})
.catch(error => {
  console.error('Error:', error);
});



{
  lessons: [
    {
      pk: '1',
      url: 'https://vimeo.com/933477203',
      title: 'Bài 1: Giới thiệu nội dung khóa học',
      description: 'A. Giới thiệu về Yoga NidraA. Giới thiệu về Yoga Nidra',
      duration: '02:09'
    },
    {
      pk: '2',
      url: 'https://vimeo.com/933550516',
      title: 'Bài 2: Yoga Nidra là gì?',
      description: 'A. Giới thiệu về Yoga Nidra',
      duration: '01:41'
    },
    {
      pk: '3',
      url: 'https://vimeo.com/933568295',
      title: 'Bài 3: Thư giãn đúng là như thế nào',
      description: null,
      duration: '01:00'
    },
    {
      pk: '4',
      url: 'https://vimeo.com/938148609',
      title: 'Bài 4: Lợi ích của Yoga Nidra với thân tâm',
      description: 'Tìm hiểu lợi ích của Yoga Nidra với thân và tâm của chúng ta như thế nào cùng với cô Opame',
      duration: '01:21'
    },
    {
      pk: '5',
      url: 'https://vimeo.com/938148961',
      title: 'Bài 5: Lợi ích của Yoga Nidra với trí',
      description: null